# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/smaharx/ml-engineering-playground/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

I use a **Random Forest classifier** for the content-refresh lane.

A Random Forest fits this problem because the data contains a mixture of numeric and categorical content/search features, and the relationship between these signals and content decline may not be linear. It also provides a probability-like score that can be used to rank pages for human review.

The goal is not to automatically change content. The goal is to produce a directional decision-support signal for prioritising which pages should be inspected first.

In [12]:
# ============================================================
# ML-08 — Setup + Dataset
# ============================================================

import os
import io
import requests
import numpy as np
import pandas as pd

from IPython.display import display

# Exact public GitHub raw file
DATA_URL = (
    "https://raw.githubusercontent.com/"
    "smaharx/ml-engineering-playground/main/"
    "data/raw/content_refresh_anonymized.csv"
)

response = requests.get(DATA_URL, timeout=60)
response.raise_for_status()

df = pd.read_csv(io.BytesIO(response.content))

print("Dataset loaded successfully.")
print("Rows:", len(df))
print("Columns:", len(df.columns))

display(pd.DataFrame({
    "column": df.columns,
    "dtype": df.dtypes.astype(str).values
}))

print("\nFirst 5 rows:")
display(df.head())

Dataset loaded successfully.
Rows: 30000
Columns: 44


,column,dtype
0,content_id,object
1,client_id,object
2,search_volume,float64
3,competition,float64
4,competition_level,object
5,cpc,float64
6,content_type,object
7,main_intent,object
8,word_count,float64
9,char_count,float64



First 5 rows:


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


In [13]:
# ============================================================
# Prepare target and grouping information
# ============================================================

print("Available columns:")
print(df.columns.tolist())

# Target
if "trend_direction" not in df.columns:
    raise ValueError(
        "Expected 'trend_direction' column was not found. "
        "Check the printed column list above."
    )

df = df[df["trend_direction"].notna()].copy()

df["is_declining"] = (
    df["trend_direction"]
    .astype(str)
    .str.strip()
    .str.lower()
    .eq("down")
    .astype(int)
)

# Remove rows with no positive impression history if available
if "impressions_90d" in df.columns:
    df = df[df["impressions_90d"].fillna(0) > 0].copy()

# Keep mature content if age is available
if "content_age_days" in df.columns:
    df = df[df["content_age_days"].fillna(0) >= 90].copy()

# Remove duplicate content IDs if available
if "content_id" in df.columns:
    df = df.drop_duplicates("content_id").copy()

# Find a grouping column.
possible_group_columns = [
    "client_id",
    "client",
    "client_name",
    "brand_id",
    "brand",
    "account_id",
]

group_col = None

for col in possible_group_columns:
    if col in df.columns:
        group_col = col
        break

if group_col is None:
    raise ValueError(
        "No client/group column was found. Available columns are:\n"
        + "\n".join(df.columns.tolist())
    )

print("Grouping column:", group_col)
print("Modeling rows:", len(df))
print("Declining rows:", int(df["is_declining"].sum()))
print("Declining rate:", round(df["is_declining"].mean(), 4))

Available columns:
['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']
Grouping column: client_id
Modeling rows: 30000
Declining rows: 16262
Declining rate: 0.5421


## 2. Split design

I use a **client-grouped holdout** rather than randomly splitting individual rows.

This is more honest for the content-refresh question because pages from the same client can share patterns. Keeping entire clients together prevents the same client's rows from appearing in both training and test data.

I hold out 20% of the available clients for testing, using a fixed random seed of 42 so the split is reproducible.

In [14]:
# Create the binary decline label from the repository's trend label.
df = df[df["trend_direction"].notna()].copy()
df["is_declining"] = (df["trend_direction"].str.lower() == "down").astype(int)

# Basic public-safe filtering used for the modeling lane.
if "impressions_90d" in df.columns:
    df = df[df["impressions_90d"].fillna(0) > 0].copy()

if "content_age_days" in df.columns:
    df = df[df["content_age_days"].fillna(0) >= 90].copy()

if "content_id" in df.columns:
    df = df.drop_duplicates("content_id").copy()

print("Modeling rows:", len(df))
print("Declining rate:", round(df["is_declining"].mean(), 3))

Modeling rows: 30000
Declining rate: 0.542


## 3. Train + compare vs my baseline

The baseline is the same simple majority-class baseline used for comparison in the modeling workflow.

The Random Forest uses a conservative feature boundary. Direct label-derived fields such as `trend_direction` and `trend_pct` are excluded. Current 90-day performance and current engagement aggregates are also excluded because they may overlap the period used to define the decline label.

The primary comparison uses the same client-grouped holdout and reports accuracy, precision, recall, F1, ROC AUC, and average precision.

In [15]:
# Conservative feature boundary.
numeric_features = [
    "search_volume",
    "competition",
    "cpc",
    "word_count",
    "char_count",
    "impressions_prev_30d",
    "clicks_prev_30d",
    "sessions_prev_30d",
    "content_age_days",
    "days_since_last_update",
]

categorical_features = [
    "competition_level",
    "content_type",
    "main_intent",
    "age_tier",
    "freshness_tier",
    "word_count_tier",
    "impression_tier",
    "position_tier",
]

numeric_features = [c for c in numeric_features if c in df.columns]
categorical_features = [c for c in categorical_features if c in df.columns]

feature_cols = numeric_features + categorical_features

X = df[feature_cols].copy()
y = df["is_declining"].copy()

# Find a client grouping column without using it as a predictive feature.
possible_group_cols = ["client_id", "client", "client_name", "brand_id", "brand"]
group_col = next((c for c in possible_group_cols if c in df.columns), None)

if group_col is None:
    raise ValueError(
        "No client grouping column was found. Check the dataset's client/group column."
    )

groups = df[group_col]

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(splitter.split(X, y, groups=groups))

X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]
y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

train_groups = groups.iloc[train_idx]
test_groups = groups.iloc[test_idx]

print("Train rows:", len(X_train))
print("Test rows:", len(X_test))
print("Held-out clients:", test_groups.nunique())
print("Train decline rate:", round(y_train.mean(), 3))
print("Test decline rate:", round(y_test.mean(), 3))

Train rows: 23837
Test rows: 6163
Held-out clients: 7
Train decline rate: 0.55
Test decline rate: 0.511


In [16]:
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features),
])

model = RandomForestClassifier(
    n_estimators=200,
    max_depth=10,
    min_samples_leaf=25,
    class_weight="balanced_subsample",
    random_state=42,
    n_jobs=-1,
)

pipeline = Pipeline([
    ("preprocess", preprocessor),
    ("model", model),
])

pipeline.fit(X_train, y_train)

model_pred = pipeline.predict(X_test)
model_prob = pipeline.predict_proba(X_test)[:, 1]

# Majority-class baseline learned from the training split.
majority_class = int(y_train.mean() >= 0.5)
baseline_pred = np.full(len(y_test), majority_class)
baseline_prob = np.full(len(y_test), y_train.mean())

results = pd.DataFrame([
    {
        "model": "Majority baseline",
        "accuracy": accuracy_score(y_test, baseline_pred),
        "precision": precision_score(y_test, baseline_pred, zero_division=0),
        "recall": recall_score(y_test, baseline_pred, zero_division=0),
        "f1": f1_score(y_test, baseline_pred, zero_division=0),
        "roc_auc": roc_auc_score(y_test, baseline_prob),
        "average_precision": average_precision_score(y_test, baseline_prob),
    },
    {
        "model": "Random Forest",
        "accuracy": accuracy_score(y_test, model_pred),
        "precision": precision_score(y_test, model_pred, zero_division=0),
        "recall": recall_score(y_test, model_pred, zero_division=0),
        "f1": f1_score(y_test, model_pred, zero_division=0),
        "roc_auc": roc_auc_score(y_test, model_prob),
        "average_precision": average_precision_score(y_test, model_prob),
    },
])

display(results.round(3))

,model,accuracy,precision,recall,f1,roc_auc,average_precision
0,Majority baseline,0.511,0.511,1.000,0.676,0.50,0.511
1,Random Forest,0.634,0.630,0.689,0.658,0.68,0.627


## 4. Errors and interpretation

The model is not perfect: some pages receive a high decline probability even though they are not labelled as declining, while some labelled declining pages receive lower scores.

The score should therefore be interpreted as a ranking signal rather than a definitive explanation of performance.

The most important interpretation is that the model combines content/search context and previous-period performance to identify patterns associated with the observed decline label. It does not establish why a page declined and does not prove that a refresh will improve future performance.

In [17]:
error_analysis = X_test.copy()
error_analysis["actual"] = y_test.values
error_analysis["predicted"] = model_pred
error_analysis["score"] = model_prob

error_analysis["error_type"] = np.select(
    [
        (error_analysis["actual"] == 1) & (error_analysis["predicted"] == 0),
        (error_analysis["actual"] == 0) & (error_analysis["predicted"] == 1),
    ],
    [
        "False negative",
        "False positive",
    ],
    default="Correct",
)

print("Error counts:")
print(error_analysis["error_type"].value_counts())

print("\nHighest-scored test examples:")
display(
    error_analysis
    .sort_values("score", ascending=False)
    .head(10)
)

Error counts:
error_type
Correct           3910
False positive    1274
False negative     979
Name: count, dtype: int64

Highest-scored test examples:


,search_volume,competition,cpc,word_count,char_count,impressions_prev_30d,clicks_prev_30d,sessions_prev_30d,content_age_days,days_since_last_update,...,main_intent,age_tier,freshness_tier,word_count_tier,impression_tier,position_tier,actual,predicted,score,error_type
12069,20.0,0.00,0.00,1516.0,8993.0,507,1,1,280,104,...,commercial,181-365,91-180,1000-2000,moderate,page_3_5,0,1,0.903152,False positive
22730,20.0,0.52,0.06,1432.0,9183.0,296,0,4,280,104,...,commercial,181-365,91-180,1000-2000,moderate,striking,1,1,0.899404,Correct
7449,210.0,0.00,0.00,1586.0,9859.0,398,0,1,271,104,...,transactional,181-365,91-180,1000-2000,moderate,page_3_5,0,1,0.899209,False positive
10080,20.0,0.01,0.00,1592.0,10156.0,335,0,1,238,103,...,commercial,181-365,91-180,1000-2000,moderate,page_3_5,0,1,0.898343,False positive
22508,40.0,0.00,0.00,1510.0,9140.0,260,0,1,280,104,...,transactional,181-365,91-180,1000-2000,moderate,page_3_5,1,1,0.896233,Correct
6228,30.0,0.00,0.00,1353.0,8916.0,744,0,1,275,104,...,informational,181-365,91-180,1000-2000,moderate,page_3_5,1,1,0.893284,Correct
5399,10.0,0.53,0.00,1535.0,9078.0,371,0,0,280,104,...,transactional,181-365,91-180,1000-2000,moderate,page_3_5,0,1,0.892888,False positive
22526,20.0,0.00,0.00,1480.0,8790.0,867,1,2,280,104,...,transactional,181-365,91-180,1000-2000,good,page_3_5,0,1,0.889621,False positive
13929,20.0,0.02,1.08,1538.0,9553.0,408,0,1,280,104,...,commercial,181-365,91-180,1000-2000,moderate,striking,1,1,0.885487,Correct
2934,10.0,0.77,0.73,1495.0,9852.0,364,0,1,300,104,...,commercial,181-365,91-180,1000-2000,moderate,page_3_5,1,1,0.884204,Correct


# ML-12 — Closing Story

## 5-Minute Demo Outline

### 1. Question — ~45 seconds
Can a leakage-aware machine learning model provide a useful directional signal for prioritising content that may need human review?

### 2. Method — ~1 minute
I used the FlyRank ML Internship content-refresh dataset, defined declining content from the available trend label, and trained a Random Forest using a conservative feature boundary.

Validation used a client-grouped holdout so entire clients were kept out of the training data.

### 3. One Chart — ~1 minute
Show the model's ranked-score distribution or evaluation chart.

Explain that the model produces a directional score for deciding what to inspect first.

### 4. Honest Result — ~1 minute
The Random Forest showed stronger discrimination than the majority baseline on the held-out client-grouped test set.

The measured result should be presented as evidence from this dataset and validation design, not as proof of causality.

### 5. Recommendation — ~45 seconds
Use the ranked output as a human-review queue.

Prioritise high-signal content for refresh review or performance review, while lower-signal content can remain under monitoring.

The model decides what to inspect first — not what must be changed.

---

## Social Post

I built a leakage-aware content opportunity scoring workflow using the FlyRank ML Internship dataset.

Instead of treating ML output as an automatic SEO decision, I used client-grouped validation and a conservative feature boundary to produce a directional ranking for human review. The Random Forest showed stronger discrimination than the majority baseline on the held-out test set.

The important takeaway: ML can help decide what to inspect first without pretending it can prove why content declined or that a refresh will cause better performance.

---

## Employer-Facing Summary

I built a leakage-aware content opportunity scoring model that ranks content for human review.

It was trained and evaluated on the FlyRank ML Internship content-refresh dataset using a client-grouped holdout and a conservative feature boundary designed to reduce temporal-overlap risk.

The Random Forest showed stronger discrimination than the majority baseline under this validation design, producing a practical directional signal for prioritising content review.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.